<a href="https://colab.research.google.com/github/AvantiShri/gotpsi_sequentialcard/blob/main/notebooks/00_GotPsi_SequentialCard_ReorganizeData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get the data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/Dean_SequentialCard

 cardS16.zip			   'raw cardsequence.zip'
'Dean SequentialCard README.gdoc'   reorganized_data


In [10]:
!md5sum "/content/drive/MyDrive/Dean_SequentialCard/cardS16.zip" #md5 checksum to compare integrity of file

29899584767d64ec6d3cfcd7496d751a  /content/drive/MyDrive/Dean_SequentialCard/cardS16.zip


In [3]:
!unzip -q "/content/drive/MyDrive/Dean_SequentialCard/cardS16.zip" -d /content/cardsequence #takes about a minute

In [4]:
#The data for the first two quarters of 2002 is hiding in cardS02Q1.zip and cardS02Q2.zip, so we will further
# unzip those
!unzip -q /content/cardsequence/cardS02/cardS02Q1.zip -d /content/cardsequence/cardS02
!unzip -q /content/cardsequence/cardS02/cardS02Q2.zip -d /content/cardsequence/cardS02

# Install dependencies

In [5]:
! pip uninstall -y gotpsi_sequentialcard
! pip install git+https://github.com/AvantiShri/gotpsi_sequentialcard.git

  Cloning https://github.com/AvantiShri/gotpsi_sequentialcard.git to /tmp/pip-req-build-u607sy12
  Running command git clone --filter=blob:none --quiet https://github.com/AvantiShri/gotpsi_sequentialcard.git /tmp/pip-req-build-u607sy12
  Resolved https://github.com/AvantiShri/gotpsi_sequentialcard.git to commit 1ecae9334d446ac4d749c0b533da81e21947f543
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gotpsi_sequentialcard: filename=gotpsi_sequentialcard-0.1.0-py3-none-any.whl size=2822 sha256=dd2c27e8181ea773d720f1e52d8e77221590701e9017291cf846a17b8ea61737
  Stored in directory: /tmp/pip-ephem-wheel-cache-5ml3_ss6/wheels/da/7d/d8/9daffca3dce198e9fd8b21af1736b50aef399566987d254017
Successfully built gotpsi_sequentialcard


In [6]:
from importlib import reload
import gotpsi_sequentialcard.utils
reload(gotpsi_sequentialcard.utils)

<module 'gotpsi_sequentialcard.utils' from '/usr/local/lib/python3.12/dist-packages/gotpsi_sequentialcard/utils.py'>

In [7]:
from gotpsi_sequentialcard.utils import find_missing_dates

missing_ranges, earliest_date, latest_date, available_dates = find_missing_dates("/content/cardsequence")

print(f"Data spans: {earliest_date} to {latest_date}")
print(f"\n{len(missing_ranges)} missing date range(s):\n")

for start, end in missing_ranges:
    if start == end:
        print(f"  {start}")
    else:
        days = (end - start).days + 1
        print(f"  {start} to {end}  ({days} days)")

Data spans: 2002-01-01 to 2018-12-29

6 missing date range(s):

  2015-08-13 to 2015-09-20  (39 days)
  2015-09-24 to 2015-09-27  (4 days)
  2015-09-29 to 2015-10-03  (5 days)
  2015-10-07 to 2015-10-09  (3 days)
  2015-10-11 to 2015-10-26  (16 days)
  2015-12-04 to 2016-05-12  (161 days)


In [8]:
from collections import namedtuple
from datetime import datetime
import os

Trial = namedtuple('Trial', ['userid', 'trial', 'target_card', 'click_sequence', 'image', 'timestamp'])

# Completed trial format:
#   userid, trial, steps, r1,r2,r3,r4,r5,r6, imagepath, timestamp
# Example: rada, 1, 2, 0,5,4,0,0,0, ../../bi/images4/c9.jpg, Thu Jan  1 00:05:52 2009
def parse_trial_file(filepath):
    trials = []
    error_lines = []

    with open(filepath, 'r', encoding='latin-1') as f:
        try:
            for line_num, line in enumerate(f, 1):
                if 'images/' not in line and "/images" not in line:
                    continue

                parts = [p.strip() for p in line.split(', ')]
                assert len(parts) == 6, f"Expected 6 parts, got {len(parts)}"

                userid = parts[0]
                if (parts[1]==""):
                    raise RuntimeError("Missing trial number")
                trial_num = int(parts[1])
                steps = int(parts[2])
                clicks = parts[3].split(",")
                imagepath = parts[4]
                timestamp_str = parts[5].strip()
                timestamp = datetime.strptime(timestamp_str, '%a %b %d %H:%M:%S %Y')

                image = os.path.basename(imagepath)

                assert clicks[0] == "0", (
                    f"Expected r1=0, got {clicks[0]}"
                )

                trailing = clicks[steps + 1:]
                assert len(trailing) == 0 or all(r == "0" for r in trailing), (
                    f"Expected trailing zeros after step {steps}, got {clicks}"
                )

                click_sequence = [int(x) for x in clicks[1:steps + 1] if x != '']
                #if (len(click_sequence) < steps):
                #  print(f"Skipped empty entries in click sequence for {line.rstrip()}")
                assert len(set(click_sequence))==len(click_sequence), (
                    f"Expected unique click sequence, got {click_sequence}")
                target_card = click_sequence[-1]

                trials.append(Trial(userid, trial_num, target_card, click_sequence, image, timestamp))

        except Exception as e:
            print(f"Error in {filepath}, line {line_num}: {e}")
            print(f"  Content: {line.rstrip()}")
            error_lines.append(line)

    return trials, error_lines

In [15]:
import csv
import re
from collections import defaultdict
from tqdm import tqdm

OUTPUT_BASE = '/content/reorganized_data'
BY_DATE_DIR = os.path.join(OUTPUT_BASE, 'by_date')
BY_USER_DIR = os.path.join(OUTPUT_BASE, 'by_user')

os.makedirs(BY_DATE_DIR, exist_ok=True)
os.makedirs(BY_USER_DIR, exist_ok=True)

HEADER = ['userid', 'trial', 'target_card', 'number_of_steps',
          'step1', 'step2', 'step3', 'step4', 'step5', 'image', 'timestamp']

def trial_to_row(trial):
    seq = trial.click_sequence
    assert len(seq) <= 5, (
        f"Click sequence has {len(seq)} steps (max 5): {seq}"
    )
    padded = list(seq) + [0] * (5 - len(seq)) #pad with zeros for shorter seqs
    return [
        trial.userid,
        trial.trial,
        trial.target_card,
        len(seq),
        padded[0], padded[1], padded[2], padded[3], padded[4],
        trial.image,
        trial.timestamp.strftime('%Y-%m-%d %H:%M:%S')
    ]

def sanitize_filename(name):
    """Replace any non-alphanumeric characters (except - and _) with _"""
    #also truncate the name to the first 50 characters
    name = name[:50]
    return re.sub(r'[^\w\-]', '_', name)

# Dictionary of open file handles per user
user_file_handles = {}
user_writers = {}
date_mismatch_count = 0
all_error_lines = []
total_trials = 0
num_trials_successful_on_first_click = 0
target_card_numbers = [0]*5

try:
    for available_date in tqdm(sorted(available_dates)):
        dat_filepath = (f"/content/cardsequence/cardS{available_date.strftime('%y')}/"
                        f"cardS{available_date.strftime('%y%m%d')}.dat")

        if ("020129" in dat_filepath) or ("020130" in dat_filepath) or ("020131" in dat_filepath):
            print(f"Skipping {available_date} because of confusion regarding which file to use")
            continue

        trials, error_lines = parse_trial_file(dat_filepath)
        assert len(trials) > 0, dat_filepath
        total_trials += len(trials)
        all_error_lines.extend(error_lines)

        #Also compile some basic stats
        for trial in trials:
            if len(trial.click_sequence)==1:
                num_trials_successful_on_first_click += 1
            target_card_numbers[trial.target_card-1] += 1

        # Write by_date file
        date_str = available_date.strftime('%Y%m%d')
        date_csv_path = os.path.join(BY_DATE_DIR, f'{date_str}.csv')

        with open(date_csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(HEADER)
            for trial in trials:
                if trial.timestamp.date() != available_date:
                    print(f"Date mismatch: file date {available_date}, "
                          f"trial timestamp {trial.timestamp.date()}, "
                          f"user {trial.userid}, trial {trial.trial}")
                    date_mismatch_count += 1
                writer.writerow(trial_to_row(trial))

        # Append to per-user files
        for trial in trials:
            uid = trial.userid
            if uid not in user_file_handles:
                safe_name = sanitize_filename(uid)
                #if (safe_name != uid):
                #    print(f"Replaced {uid} with {safe_name}")
                user_csv_path = os.path.join(BY_USER_DIR, f'{safe_name}.csv')
                fh = open(user_csv_path, 'w', newline='')
                user_file_handles[uid] = fh
                user_writers[uid] = csv.writer(fh)
                user_writers[uid].writerow(HEADER)
            user_writers[uid].writerow(trial_to_row(trial))

finally:
    # Flush and close all user file handles
    for uid, fh in user_file_handles.items():
        fh.flush()
        fh.close()

  0%|          | 1/5979 [00:00<17:37,  5.65it/s]

Error in /content/cardsequence/cardS02/cardS020101.dat, line 11226: Missing trial number
  Content: , , -1, ,,0,0,0,0,0,0, images/c6.gif, Tue Jan  1 22:53:29 2002
Error in /content/cardsequence/cardS02/cardS020102.dat, line 8818: Missing trial number
  Content: , , 2, 0,2,3,0,0,0, images/ok.jpg, Wed Jan  2 16:14:50 2002


  0%|          | 4/5979 [00:00<08:59, 11.06it/s]

Error in /content/cardsequence/cardS02/cardS020103.dat, line 1682: Missing trial number
  Content: oriok, , 2, 0,4,3,0,0,0, images/c3.gif, Thu Jan  3 04:23:22 2002
Error in /content/cardsequence/cardS02/cardS020104.dat, line 6132: Missing trial number
  Content: toby2, , 3, 0,3,5,4,0,0, images/c9.jpg, Fri Jan  4 10:31:21 2002


  0%|          | 8/5979 [00:00<08:49, 11.27it/s]

...

  1%|▏         | 81/5979 [00:08<13:27,  7.30it/s]

Error in /content/cardsequence/cardS02/cardS020322.dat, line 19573: Missing trial number
  Content: test4, , 6, ,,2,3,4,5,1, ../../bi/images4/ok.jpg, Fri Mar 22 16:18:40 2002


 97%|█████████▋| 5781/5979 [29:40<00:55,  3.59it/s]

Error in /content/cardsequence/cardS18/cardS180614.dat, line 8939: Missing trial number
  Content: lweltzer, , 0, ,2,0,0,0,0,0, ../images4/c3.jpg, Thu Jun 14 19:50:55 2018
Error in /content/cardsequence/cardS18/cardS180615.dat, line 672: Missing trial number
  Content: conan01, , 0, ,3,0,0,0,0,0, ../images4/c4.gif, Fri Jun 15 06:41:00 2018


100%|██████████| 5979/5979 [29:59<00:00,  3.32it/s]


In [16]:
print(f"\nTotal trials: {total_trials}")
print(f"Fraction of trials successful on first click: {num_trials_successful_on_first_click/total_trials}")
print(f"Target card proportions: {[x/total_trials for x in target_card_numbers]}")
print(f"Total date mismatches: {date_mismatch_count}")
print(f"Number of error lines: {len(all_error_lines)}")
print(f"Total user files written: {len(user_file_handles)}")


Total trials: 36903314
Fraction of trials successful on first click: 0.20041682435349845
Target card proportions: [0.1999168692546149, 0.2001067438008413, 0.20008002533322616, 0.1999909276440593, 0.19990543396725832]
Total date mismatches: 0
Number of error lines: 69
Total user files written: 115044


In [17]:
!du -h reorganized_data

2.0G	reorganized_data/by_date
2.3G	reorganized_data/by_user
4.2G	reorganized_data


In [18]:
!tar -czf reorganized_data.tar.gz reorganized_data

In [19]:
!du -h reorganized_data.tar.gz

600M	reorganized_data.tar.gz


In [20]:
!md5sum reorganized_data.tar.gz
!cp reorganized_data.tar.gz /content/drive/MyDrive/Dean_SequentialCard/reorganized_data.tar.gz

bef90a40e9f360a11ad4b6cff91354da  reorganized_data.tar.gz


In [14]:
#!rm -rf reorganized_data

In [21]:
!cat /content/cardsequence/cardS04/cardS040527.dat | grep -a -n "Gualter" | head #the name that was so long I couldn't turn it into a file name

4932:    Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is very displeased.Gualter was banished of the group by Christ 31 and for the moderator of the group and him it is